# Análisis de hábitos musicales: Springfield vs Shelbyville

**Autor:** Rubén Montaño  
**Fecha:** 2026-05-2026

**Herramientas:** Python, pandas  
**Dataset:** Registros de reproducción musical de dos ciudades

## Contexto del proyecto

Una plataforma de streaming musical quiere entender si los hábitos de escucha de sus usuarios varían según la ciudad y el día de la semana. Para responder esta pregunta, se analizaron más de 65,000 registros de reproducción de dos ciudades: **Springfield** y **Shelbyville**.

### Hipótesis a validar

> *¿El comportamiento de los usuarios —en cuanto a la música que escuchan— varía según la ciudad y el día de la semana?*

### Enfoque

El análisis sigue tres fases: exploración inicial de los datos, preprocesamiento (limpieza de nombres, nulos y duplicados) y análisis comparativo entre ciudades y días de la semana.

## Tabla de contenido

1. [Carga y exploración inicial](#scrollTo=pCOX42vSYgRD)
2. [Preprocesamiento de datos](#scrollTo=munZuJRjYgRG)
   - Normalización de columnas
   - Tratamiento de valores nulos
   - Eliminación de duplicados
3. [Análisis comparativo](#scrollTo=fMSVcoqUYgRI)
   - Reproducciones por ciudad
   - Reproducciones por día
   - Cruce: ciudad × día
4. [Conclusiones](#scrollTo=S1T-hn0ZYgRN)

## 1. Carga y exploración inicial <a id='carga'></a>

In [17]:
import pandas as pd

Se preserva una copia del dataset original (`df_raw`) como respaldo inmutable. Todo el trabajo de limpieza y análisis se realiza sobre `df`, la copia de trabajo.

In [18]:
df_raw = pd.read_csv('/content/music_project_en.csv')
df = df_raw.copy()

Una vista rápida de los primeros registros para familiarizarnos con la estructura:

In [19]:
df.head(10)

,userID,Track,artist,genre,City,time,Day
0,FFB692EC,Kamigata To Boots,The Mass Missile,rock,Shelbyville,20:28:33,Wednesday
1,55204538,Delayed Because of Accident,Andreas Rönnberg,rock,Springfield,14:07:09,Friday
2,20EC38,Funiculì funiculà,Mario Lanza,pop,Shelbyville,20:58:07,Wednesday
3,A3DD03C9,Dragons in the Sunset,Fire + Ice,folk,Shelbyville,08:37:09,Monday
4,E2DC1FAE,Soul People,Space Echo,dance,Springfield,08:34:34,Monday
5,842029A1,Chains,Obladaet,rusrap,Shelbyville,13:09:41,Friday
6,4CB90AA5,True,Roman Messer,dance,Springfield,13:00:07,Wednesday
7,F03E1C1F,Feeling This Way,Polina Griffith,dance,Springfield,20:47:49,Wednesday
8,8FA1D3BE,L’estate,Julia Dalia,ruspop,Springfield,09:17:40,Friday
9,E772D5C0,Pessimist,NaN,dance,Shelbyville,21:20:49,Wednesday


In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65079 entries, 0 to 65078
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0     userID  65079 non-null  object
 1   Track     63736 non-null  object
 2   artist    57512 non-null  object
 3   genre     63881 non-null  object
 4     City    65079 non-null  object
 5   time      65079 non-null  object
 6   Day       65079 non-null  object
dtypes: object(7)
memory usage: 3.5+ MB


### Observaciones iniciales

- El dataset contiene **65,079 registros** con 7 columnas, todas de tipo `object`.
- Hay **valores nulos** en tres columnas clave: `Track` (1,343), `artist` (7,567) y `genre` (1,198).
- Los **nombres de columnas** presentan inconsistencias: mezcla de mayúsculas/minúsculas y espacios sobrantes (`'  userID'`, `'  City  '`).
- La columna `time` es string; para este análisis no se requiere conversión a datetime, pero sería necesario si se analizaran patrones por hora.

## 2. Preprocesamiento de datos <a id='preprocesamiento'></a>

### 2.1 Normalización de columnas <a id='columnas'></a>

Se estandarizan todos los nombres a formato `snake_case`: minúsculas, sin espacios sobrantes y con guiones bajos entre palabras.

In [21]:
print('Antes:', df.columns.tolist())

df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df.rename(columns={'userid': 'user_id'}, inplace=True)

print('Después:', df.columns.tolist())

Antes: ['  userID', 'Track', 'artist', 'genre', '  City  ', 'time', 'Day']
Después: ['user_id', 'track', 'artist', 'genre', 'city', 'time', 'day']


### 2.2 Tratamiento de valores nulos <a id='nulos'></a>

Se cuantifican los nulos por columna antes de decidir la estrategia de tratamiento.

In [22]:
null_pct = (df.isnull().mean() * 100).round(2)
print(null_pct[null_pct > 0])

track      2.06
artist    11.63
genre      1.84
dtype: float64


Los nulos se concentran en campos de metadatos de la canción (`track`, `artist`, `genre`). Dado que no es posible recuperar esta información y que representan una proporción manejable del total, se imputan con el valor `'unknown'` para no perder los registros asociados.

In [23]:
columns_to_replace = ['track', 'artist', 'genre']
for column in columns_to_replace:
    df[column] = df[column].fillna('unknown')

# Verificación: ya no quedan nulos
print(df.isnull().sum())

user_id    0
track      0
artist     0
genre      0
city       0
time       0
day        0
dtype: int64


### 2.3 Eliminación de duplicados <a id='duplicados'></a>

#### Duplicados explícitos

Se identifican y eliminan filas completamente idénticas.

In [24]:
print(f'Duplicados encontrados: {df.duplicated().sum():,}')

df = df.drop_duplicates().reset_index(drop=True)

print(f'Duplicados tras limpieza: {df.duplicated().sum()}')

Duplicados encontrados: 3,826
Duplicados tras limpieza: 0


#### Validación post-limpieza

Se verifica el impacto de la eliminación de duplicados sobre el volumen total de datos para asegurar que no se esté perdiendo una proporción crítica de registros.

In [25]:
print(f'Filas originales:          {len(df_raw):,}')
print(f'Filas después de limpieza: {len(df):,}')
print(f'Eliminadas:                {len(df_raw) - len(df):,} ({(len(df_raw) - len(df)) / len(df_raw) * 100:.1f}%)')

Filas originales:          65,079
Filas después de limpieza: 61,253
Eliminadas:                3,826 (5.9%)


La pérdida de datos es del **5.9%**, dentro de un rango aceptable que no compromete la representatividad del análisis.

#### Duplicados implícitos

Se revisan los valores únicos de `genre` para detectar variantes del mismo género escritas de forma diferente.

In [26]:
print(sorted(df['genre'].unique()))

['acid', 'acoustic', 'action', 'adult', 'africa', 'afrikaans', 'alternative', 'ambient', 'americana', 'animated', 'anime', 'arabesk', 'arabic', 'arena', 'argentinetango', 'art', 'audiobook', 'avantgarde', 'axé', 'baile', 'balkan', 'beats', 'bigroom', 'black', 'bluegrass', 'blues', 'bollywood', 'bossa', 'brazilian', 'breakbeat', 'breaks', 'broadway', 'cantautori', 'cantopop', 'canzone', 'caribbean', 'caucasian', 'celtic', 'chamber', 'children', 'chill', 'chinese', 'choral', 'christian', 'christmas', 'classical', 'classicmetal', 'club', 'colombian', 'comedy', 'conjazz', 'contemporary', 'country', 'cuban', 'dance', 'dancehall', 'dancepop', 'dark', 'death', 'deep', 'deutschrock', 'deutschspr', 'dirty', 'disco', 'dnb', 'documentary', 'downbeat', 'downtempo', 'drum', 'dub', 'dubstep', 'eastern', 'easy', 'electronic', 'electropop', 'emo', 'entehno', 'epicmetal', 'estrada', 'ethnic', 'eurofolk', 'european', 'experimental', 'extrememetal', 'fado', 'film', 'fitness', 'flamenco', 'folk', 'folklor

Se detectan tres variantes del género hip-hop: `'hip'`, `'hop'` y `'hip-hop'`, que deben unificarse bajo `'hiphop'`.

In [27]:
def replace_wrong_values(df, column, wrong_values, correct_value):
    """Unifica valores duplicados implícitos en una columna categórica."""
    mapping = {val: correct_value for val in wrong_values}
    df[column] = df[column].replace(mapping)
    return df

df = replace_wrong_values(df, 'genre', ['hip', 'hop', 'hip-hop'], 'hiphop')

Verificación: las variantes ya no aparecen en los valores únicos.

In [28]:
# Confirmar que 'hip', 'hop' y 'hip-hop' ya no existen
hiphop_check = [g for g in df['genre'].unique() if 'hip' in g or 'hop' in g]
print(f'Variantes de hip-hop restantes: {hiphop_check}')

Variantes de hip-hop restantes: ['hiphop', 'triphop']


## 3. Análisis comparativo <a id='analisis'></a>

### 3.1 Reproducciones por ciudad <a id='por-ciudad'></a>

In [29]:
print(df.groupby('city')['track'].count())

city
Shelbyville    18512
Springfield    42741
Name: track, dtype: int64


Springfield acumula **42,741** reproducciones frente a **18,512** de Shelbyville, es decir, Springfield registra **2.3 veces más actividad**. Una hipótesis razonable es que esta diferencia refleja la proporción de población entre ambas ciudades (Springfield tiene aproximadamente 3 veces más habitantes que Shelbyville).

### 3.2 Reproducciones por día <a id='por-dia'></a>

In [30]:
print(df.query("day in ['Monday', 'Friday']").groupby('day')['track'].count())

day
Friday    21840
Monday    21354
Name: track, dtype: int64


In [31]:
reps_lunes = df.query("day == 'Monday'")['track'].count()
reps_viernes = df.query("day == 'Friday'")['track'].count()
print(f'Diferencia lunes vs viernes: {reps_viernes - reps_lunes:,} reproducciones ({(reps_viernes - reps_lunes) / reps_viernes * 100:.1f}%)')

Diferencia lunes vs viernes: 486 reproducciones (2.2%)


La diferencia entre viernes (21,840) y lunes (21,354) es de apenas **486 reproducciones (2.2%)**, una variación que no es significativa.

### 3.3 Cruce: ciudad × día <a id='cruce'></a>

Para un análisis más granular, se cruzan ambas dimensiones: ciudad y día de la semana.

In [32]:
def number_tracks(df, day, city):
    """Cuenta reproducciones filtradas por día y ciudad."""
    df_filtered = df[(df['day'] == day) & (df['city'] == city)]
    return df_filtered['user_id'].count()

# Tabla resumen
results = pd.DataFrame({
    'Ciudad': ['Springfield', 'Springfield', 'Shelbyville', 'Shelbyville'],
    'Día': ['Monday', 'Friday', 'Monday', 'Friday'],
    'Reproducciones': [
        number_tracks(df, 'Monday', 'Springfield'),
        number_tracks(df, 'Friday', 'Springfield'),
        number_tracks(df, 'Monday', 'Shelbyville'),
        number_tracks(df, 'Friday', 'Shelbyville'),
    ]
})

print(results.to_string(index=False))

     Ciudad    Día  Reproducciones
Springfield Monday           15740
Springfield Friday           15945
Shelbyville Monday            5614
Shelbyville Friday            5895


En ambas ciudades, la actividad es muy similar entre lunes y viernes. La diferencia dominante es entre ciudades, no entre días.

## 4. Conclusiones <a id='conclusiones'></a>

### Sobre la hipótesis

Los datos muestran que **la ciudad sí influye en el volumen de reproducciones**, pero **el día de la semana no genera una diferencia relevante**.

- **Ciudad:** Springfield registra 2.3 veces más reproducciones que Shelbyville (42,741 vs 18,512). Esta proporción es consistente con la diferencia de población entre ambas ciudades (~3:1), lo que sugiere que el volumen de escucha está más relacionado con el tamaño de la base de usuarios que con preferencias culturales distintas.
- **Día:** La diferencia entre lunes y viernes es de solo 2.2% a nivel global, y se mantiene estable al segmentar por ciudad. No hay evidencia de un cambio de comportamiento significativo entre estos dos días.

### Sobre el proceso de limpieza

- Se normalizaron los nombres de columnas (minúsculas, snake_case, sin espacios) para facilitar el manejo programático.
- Se imputaron 10,108 valores nulos en `track`, `artist` y `genre` con `'unknown'` para preservar los registros.
- Se eliminaron 3,826 duplicados explícitos (5.9% del dataset), una pérdida aceptable.
- Se corrigieron 3 variantes del género hip-hop (`hip`, `hop`, `hip-hop` → `hiphop`) mediante una función reutilizable.

### Limitaciones y trabajo futuro

- El análisis se limita a lunes y viernes; incluir los 7 días permitiría detectar patrones de fin de semana.
- No se analizó la distribución por hora del día, lo cual podría revelar diferencias en horarios pico entre ciudades.
- La columna `genre` tiene más de 260 categorías únicas; un análisis de los géneros más populares por ciudad podría enriquecer las conclusiones.
- No se normalizó la actividad por número de usuarios únicos por ciudad, lo que daría una métrica de reproducciones *per cápita* más precisa.